# What decomposing the hybrid solver into processes adds, and what it costs

_Investigation `hybrid-generalizability` — coder reproduction notebook._

**Question.** Once the co-simulation reproduces the embedded hybrid solver, what does expressing the method as interchangeable processes add (alternative PDE discretizations, alternative splitting orders, swappable engines), and what do those choices cost in accuracy and performance compared with VCell's own discretization of the same problems?

A study of the generality gained by decomposing the hybrid solver into process-bigraph processes.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-pde-particle/viva-pde-particle').is_dir():
    REPO = Path('/home/runner/work/viva-pde-particle/viva-pde-particle')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_pde_particle.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: PDE engine swap, FV (VCell FV_SOLVER scheme) vs FEniCSx Q1 under the same coupling (`fenicsx-same-problem`)

**Question.** Can the PDE half of the co-simulation be swapped for a different discretization
(FEniCSx, Q1 finite elements) without changing anything else, and does that change the
coupled result beyond discretization error?

**Objective.** Part 1: cosine-mode diffusion on N×N grids (N = 11 ... 81) for three engines (FV, Q1
lumped mass, Q1 consistent mass), each compared to the analytic decay; h-orders fitted.
Part 2: A4's A_p + B_f → C_f on the 11×11×3 slab, 16 seeds per engine with the same
seeds (common random numbers, so paired comparison); survival and produced C compared.

**Hypothesis.** FenicsxReactionDiffusion exposes the same ports as FVReactionDiffusion. On a structured
mesh whose vertices are the grid nodes, its lumped mass equals the FV control volumes, so
particle binning and units are unchanged. Q1 and FV are both second order. On matched
grids, the coupled hybrid results should agree to within sampling.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `engine-swap-bimolecular` | `viva_pde_particle.composites.examples.bimolecular_hybrid` | 0 | n_particles=20000, dt=0.01 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_pde_particle.composites.examples.bimolecular_hybrid`** — `spec_viva_pde_particle_composites_examples_bimolecular_hybrid` (a plain, editable dict)


_composite spec file for `viva_pde_particle.composites.examples.bimolecular_hybrid` not found under `viva_pde_particle/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fenicsx-same-problem ===
STUDY = 'fenicsx-same-problem'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**engine-swap**


In [ ]:
# engine-swap
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| fem-second-order | kind=derived_scalar field=b1_fem_lumped_h_order | op range low 1.8 high 2.2 provenance {'kind': 'theory', 'note': 'Q1 Lagrange elements are second order in L∞ for smooth solutions.'} |
| fv-second-order | kind=derived_scalar field=b1_fv_h_order | op range low 1.8 high 2.2 provenance {'kind': 'theory', 'note': 'Node-centred FV with a 5/7-point stencil is second order.'} |
| engine-swap-preserves-coupled-result | kind=derived_scalar field=b1_fem_lumped_vs_fv_survival_max_paired_z | op < value 4.0 provenance {'kind': 'theory', 'note': 'On matched grids the engines differ only by O(h²) discretization, far below the coupling error.'} |


## Study: Hybrid co-simulation with a body-fitted unstructured PDE mesh (ball) (`unstructured-geometry`)

**Question.** Can the co-simulation use a body-fitted unstructured mesh for the PDE half of a curved
domain, as an alternative to VCell's own Cartesian volume grid with staircase-membrane
numerics, while keeping the particle ↔ field handoff conservative and accurate?

**Objective.** Ball R = 4 µm; mesh h = 0.8 µm; background grid 33³ with Δ = 0.25 µm; dt = 0.01; 8 seeds.
- Conversion A_p → B_f (k = 0.5 /s, 20,000 A): A against exp decay, A + B conservation,
  and B against the continuum FEM solution.
- Exchange A_p ⇌ B_f with field-dependent creation inside the spherical compartment
  (k1 = 1, k2 = 0.5, B₀ = 0.2 µM): mass balance and the steady ratio k2/k1.

**Hypothesis.** The PDE runs as P1 on a gmsh tetrahedral mesh of a ball, and Smoldyn runs inside a
reflecting sphere. They couple through a fine background grid with one sparse matrix P:
fields are interpolated as P·u, and particle loads are Pᵀ·counts. P's rows sum to 1, so
the handoff conserves molecules exactly. The only residual is the mismatch between the
particle side's geometry (exact sphere) and the PDE side's (polyhedral mesh, −1.4%
volume).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `ball-conversion` | `viva_pde_particle.composites.examples.bimolecular_hybrid` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_pde_particle.composites.examples.bimolecular_hybrid`** — `spec_viva_pde_particle_composites_examples_bimolecular_hybrid` (a plain, editable dict)


_composite spec file for `viva_pde_particle.composites.examples.bimolecular_hybrid` not found under `viva_pde_particle/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: unstructured-geometry ===
STUDY = 'unstructured-geometry'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**unstructured-geometry**


In [ ]:
# unstructured-geometry
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| particle-decay-exact-in-ball | kind=derived_scalar field=b2_conversion_decay_max_z | op < value 4.0 provenance {'kind': 'theory', 'note': 'A first-order reaction is unaffected by geometry.'} |
| handoff-conserves-in-ball | kind=derived_scalar field=b2_conversion_mass_balance_max_rel_dev | op < value 0.02 provenance {'kind': 'theory', 'note': 'Pᵀ conserves counts exactly; the residual is explicit-coupling lag plus sampling.'} |
| field-matches-continuum-in-ball | kind=derived_scalar field=b2_conversion_b_vs_continuum_rel_l2 | op < value 0.05 provenance {'kind': 'theory', 'note': '20,000 particles over 643 DOFs, 8 seeds.'} |
| exchange-steady-state-in-ball | kind=derived_scalar field=b2_exchange_steady_ratio_rel_err | op < value 0.05 provenance {'kind': 'theory', 'note': 'Detailed balance; field-dependent creation inside the spherical compartment.'} |


## Study: Same ball, two discretizations: native VCell (Cartesian + membrane numerics) vs co-simulation (body-fitted P1) (`sphere-cosim-vs-native`)

**Question.** On a curved domain (a ball), how do the native VCell hybrid and the co-simulation compare?
- Native VCell uses analytic geometry on its Cartesian volume grid: the membrane is a
  staircase geometrically, but numerically uses local tangent-plane projections and
  Voronoi neighbours.
- The co-simulation uses a body-fitted P1 mesh coupled through a background grid.

**Objective.** Box [0, 9]³ with Δ = 0.25 µm (37³). Ball centre (4.5, 4.5, 4.5), R = 4 µm, 0.5 µm clear of
the box walls, so the box-wall creation defect (vcell-fvsolver#24) cannot act.
Co-sim mesh h = 0.8 µm. 8 seeds per solver.
- Conversion A_p → B_f (k = 0.5 /s, 20,000 A).
- Exchange A_p ⇌ B_f with field-dependent creation in the cell (k1 = 1, k2 = 0.5, B₀ = 0.2 µM).

**Hypothesis.** Both are consistent discretizations of the same problem, so they should agree to within
sampling plus discretization error. VCell's geometry handling is mature, and the
comparison measures which discretization represents this geometry and coupling better at
comparable resolution.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `ball-both-solvers` | `viva_pde_particle.composites.examples.bimolecular_hybrid` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_pde_particle.composites.examples.bimolecular_hybrid`** — `spec_viva_pde_particle_composites_examples_bimolecular_hybrid` (a plain, editable dict)


_composite spec file for `viva_pde_particle.composites.examples.bimolecular_hybrid` not found under `viva_pde_particle/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: sphere-cosim-vs-native ===
STUDY = 'sphere-cosim-vs-native'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**sphere-cosim-vs-native**


In [ ]:
# sphere-cosim-vs-native
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**sphere-3d-conversion**


In [ ]:
# sphere-3d-conversion
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**sphere-3d-exchange**


In [ ]:
# sphere-3d-exchange
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**sphere-3d-movie**


In [ ]:
# sphere-3d-movie
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**sphere-3d-interactive**


In [ ]:
# sphere-3d-interactive
show_viz(_render_one('threejs:viz/sphere_3d_interactive.html', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| native-represents-ball-volume | kind=derived_scalar field=b2b_native_staircase_volume_rel_err | op range low -0.02 high 0.02 provenance {'kind': 'theory', 'note': 'Δ = 0.25 µm on R = 4 µm.'} |
| conversion-agrees | kind=derived_scalar field=b2b_conversion_survival_max_z | op < value 4.0 provenance {'kind': 'theory', 'note': 'Same first-order kinetics in the same ball.'} |
| conversion-field-profiles-agree | kind=derived_scalar field=b2b_final_radial_b_rel_diff | op < value 0.05 provenance {'kind': 'theory', 'note': 'Discretization plus sampling.'} |
| exchange-agrees | kind=derived_scalar field=b2b_exchange_a_totals_max_z | op < value 4.0 provenance {'kind': 'theory', 'note': 'Kept so the open discrepancy stays visible in the report card.'} |


## Study: Fields next to a curved membrane: native VCell on pyvcell's smoothed grid vs co-sim on a body-fitted mesh (`near-membrane-fields`)

**Question.** B2b found that the native hybrid loses molecules at the curved membrane. Where does that
show up in the field, and does the co-simulation have its own boundary artifact?
Native fields are read on pyvcell's smoothed unstructured grid, so boundary elements sit
near the true membrane rather than on the staircase.

**Objective.** Same ball, grid, mesh and solvers as B2b. Conversion A_p → B_f (k = 0.5 /s, 20,000 A
uniform in the ball), t = 2 s, 32 seeds per solver.
- E[B] = A₀(1 − e^(−kt))/(V·N_A), with V each solver's own domain volume.
- The interior is r < R/2. The outer shell is r > R − 2Δ = 3.5 µm.
- Native values are volume-weighted on the smoothed cells; co-sim values are weighted by
  the lumped mass of the P1 DOFs.

**Hypothesis.** With A placed uniformly in the ball, A stays uniform in expectation, so E[B] is spatially
uniform. Any radial structure in the ensemble-mean B therefore comes from particle-to-field
transfer at the membrane.
- Native: exterior-node binning drops B sources next to the membrane, so the outer shell
  should be depleted.
- Co-sim: exterior-node counts are folded onto boundary DOFs (Pᵀ), so its outer shell
  should be slightly enriched. Study B2d later revised this cause; see the conclusion.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `ball-conversion` | `viva_pde_particle.composites.examples.bimolecular_hybrid` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_pde_particle.composites.examples.bimolecular_hybrid`** — `spec_viva_pde_particle_composites_examples_bimolecular_hybrid` (a plain, editable dict)


_composite spec file for `viva_pde_particle.composites.examples.bimolecular_hybrid` not found under `viva_pde_particle/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: near-membrane-fields ===
STUDY = 'near-membrane-fields'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**near-membrane-fields**


In [ ]:
# near-membrane-fields
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| smoothed-grid-represents-ball | kind=derived_scalar field=b2c_node_centred_smoothed_volume_rel_err | op range low -0.01 high 0.01 provenance {'kind': 'theory', 'note': 'Whole node-centred voxels; windowed-sinc smoothing preserves volume.'} |
| smoothing-improves-surface | kind=derived_scalar field=b2c_smoothed_surface_r_rms_err | op < value 0.08 provenance {'kind': 'theory', 'note': 'Smoothing should cut the staircase error (Δ = 0.25 µm) substantially.'} |
| native-interior-matches-expectation | kind=derived_scalar field=b2c_native_interior_rel | op range low -0.02 high 0.02 provenance {'kind': 'theory', 'note': 'First-order conversion; the expectation is exact.'} |
| cosim-interior-matches-expectation | kind=derived_scalar field=b2c_cosim_interior_rel | op range low -0.02 high 0.02 provenance {'kind': 'theory', 'note': 'As above.'} |
| native-shell-unbiased | kind=derived_scalar field=b2c_native_shell_bias | op range low -0.01 high 0.01 provenance {'kind': 'theory', 'note': 'E[B] is uniform; kept so the boundary artifact stays visible in the report card.'} |
| cosim-shell-unbiased | kind=derived_scalar field=b2c_cosim_shell_bias | op range low -0.01 high 0.01 provenance {'kind': 'theory', 'note': 'As above.'} |


## Study: Removing the co-simulation's boundary artifact: particle transfer, membrane geometry and curved-compartment creation (`cosim-boundary-transfer`)

**Question.** B2c found the co-simulation's ensemble-mean B 1.3% too high in the outer shell of the ball.
Two causes were candidates:
- the two-stage particle transfer (nearest grid node, then Pᵀ to mesh DOFs);
- the domain mismatch: Smoldyn's exact sphere holds about 1.4% more volume than the
  inscribed P1 mesh.
Which is it, and what does removing it cost?

**Objective.** Same ball, background grid, mesh and model as B2b/B2c; 32 seeds per variant.
Variants (particle_transfer / membrane):
- grid / sphere (B2b/B2c);
- positions / sphere;
- grid / mesh;
- positions / mesh.
Cases and measures:
- Conversion (uniform A, so E[B] is uniform): interior level against
  E[B] = A₀(1 − e^(−kt))/(V_mesh·N_A), and the outer-shell bias (r > R − 2Δ vs r < R/2).
- Exchange with field-dependent creation in the cell: final mass balance and steady ratio.
- Wall time per simulated second.

**Hypothesis.** B2c attributed the bias to the transfer, so the exact P1 load from molecule positions
should remove it.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `ball-transfer-variants` | `viva_pde_particle.composites.examples.bimolecular_hybrid` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_pde_particle.composites.examples.bimolecular_hybrid`** — `spec_viva_pde_particle_composites_examples_bimolecular_hybrid` (a plain, editable dict)


_composite spec file for `viva_pde_particle.composites.examples.bimolecular_hybrid` not found under `viva_pde_particle/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: cosim-boundary-transfer ===
STUDY = 'cosim-boundary-transfer'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**cosim-boundary-transfer**


In [ ]:
# cosim-boundary-transfer
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| mesh-membrane-removes-shell-bias | kind=derived_scalar field=b2d_grid_mesh_shell_bias | op range low -0.005 high 0.005 provenance {'kind': 'theory', 'note': 'E[B] is uniform; 32 seeds give an SE of about 0.1%.'} |
| mesh-membrane-exchange-balance | kind=derived_scalar field=b2d_grid_mesh_exchange_mass_balance_final_signed | op range low -0.015 high 0.015 provenance {'kind': 'theory', 'note': 'First-order coupling error only. The box-domain baseline (A3, two-way-exchange) is 1.2%.'} |
| sphere-membrane-shell-bias | kind=derived_scalar field=b2d_grid_sphere_shell_bias | op range low -0.005 high 0.005 provenance {'kind': 'theory', 'note': 'Kept so the artifact stays visible in the report card.'} |


## Study: The co-simulation on VCell's own geometry: staircase PDE, smooth membrane, accessible-volume correction (`vcell-geometry-cosim`)

**Question.** Native VCell solves the PDE on the voxel staircase but confines particles by a smooth membrane.
For the B2b ball the two domains differ by ~1.8% in volume. That leaves a −1.4% outer-shell
depletion and an exchange drift, even with the binning correction of vcell-fvsolver#25
(Study B2c). With the co-simulation running on exactly VCell's geometry, can an
accessible-volume correction make the coupling both unbiased and conservative?

**Objective.** The B2b ball as a vcell-fenics GeometryDescription, realized by VCell itself
(VCellFVRealization, via libvcell):
- **PDE:** FV on VCell's staircase `cell` (266.734 µm³).
- **Smoldyn:** VCell's smooth membrane (9,516 triangles enclosing 261.818 µm³) and VCell's
  compartment points.
- **Adapters:** counts at exterior nodes are folded into the domain (compartment-aware
  binning), and fields are extended into the exterior band for 0th-order creation.

32 seeds per variant.
- Conversion (k = 0.5, 20,000 A uniform in the smooth cell): interior B against
  E[B] = A₀(1 − e^(−kt))/(V_PDE·N_A), the outer-shell bias (r > 3.5 vs r < 2) and A + B balance.
- Exchange (k1 = 1, k2 = 0.5, B₀ = 0.2 µM): A + B balance and the steady ratio.

All PDE masses and averages use the PDE's own element volumes.

**Hypothesis.** Three variants of the particle → field transfer:
- **`none`** (full voxel volumes): reproduces native VCell with the #25 correction. The shell
  is depleted by ~1.4%, and exchange drifts by the smooth-vs-staircase volume mismatch.
- **`adapters`** (accessible volumes in the adapter only): removes the shell bias, but the PDE
  still integrates over full voxels, so particle sources are overcounted (~+1.4% in conversion).
- **`adapters+volumes`** (accessible volumes in the adapter and as the FV element volumes):
  unbiased and conservative.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `vcell-ball` | `viva_pde_particle.composites.examples.bimolecular_hybrid` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_pde_particle.composites.examples.bimolecular_hybrid`** — `spec_viva_pde_particle_composites_examples_bimolecular_hybrid` (a plain, editable dict)


_composite spec file for `viva_pde_particle.composites.examples.bimolecular_hybrid` not found under `viva_pde_particle/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: vcell-geometry-cosim ===
STUDY = 'vcell-geometry-cosim'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**vcell-geometry-cosim**


In [ ]:
# vcell-geometry-cosim
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**vcell-geometry-3d**


In [ ]:
# vcell-geometry-3d
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**vcell-geometry-3d-movie**


In [ ]:
# vcell-geometry-3d-movie
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**vcell-geometry-3d-interactive**


In [ ]:
# vcell-geometry-3d-interactive
show_viz(_render_one('threejs:viz/vcell_geometry_3d_interactive.html', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| corrected-shell-unbiased | kind=derived_scalar field=b2e_adapters_volumes_shell_bias | op range low -0.005 high 0.005 provenance {'kind': 'theory', 'note': 'E[B] is spatially uniform; 32 seeds give an SE of about 0.1%.'} |
| corrected-interior-matches | kind=derived_scalar field=b2e_adapters_volumes_interior_rel | op range low -0.01 high 0.01 provenance {'kind': 'theory', 'note': "First-order conversion; the expectation is exact on the PDE's own volume."} |
| corrected-exchange-conserves | kind=derived_scalar field=b2e_adapters_volumes_exchange_balance | op range low -0.015 high 0.015 provenance {'kind': 'theory', 'note': 'First-order coupling error only; the box-domain baseline (A3) is 1.2%.'} |
| vcell-like-shell-depletion | kind=derived_scalar field=b2e_none_shell_bias | op range low -0.005 high 0.005 provenance {'kind': 'theory', 'note': 'Kept so the volume mismatch stays visible in the report card.'} |


## Study: One non-convex GeometryDescription, three solver paths: native VCell, co-sim on VCell's geometry, co-sim on a Netgen mesh (`dumbbell-three-ways`)

**Question.** Phase 7e end to end on a non-sphere: does one geometry description drive the native VCell solver
and both co-simulation paths, and do the accessible-volume correction and the body-fitted mesh
stay unbiased and conservative on a thin-necked, non-convex shape?

**Objective.** A vcell-fenics GeometryDescription: two balls (R = 1.5 µm, centres 2.8 µm apart) joined by a
neck (radius 0.6 µm), in an 8 × 6 × 6 µm box, Δ = 0.25 µm (33 × 25 × 25 grid). The four paths:
- **native:** the description converted to VCML; vcell-fvsolver with embedded Smoldyn; 8 seeds.
- **vcell_none, vcell_corrected:** the co-simulation on VCell's own realization, with
  correction `none` or `adapters+volumes`; 16 seeds each.
- **mesh:** the co-simulation on a Netgen tet mesh (h = 0.3); 16 seeds.

Cases and measures:
- conversion (k = 0.5, 20,000 A uniform; 2 s): A + B balance; near-membrane bias (mean B within
  0.5 µm of the membrane against deeper B; E[B] is uniform); lobe asymmetry;
- exchange (k1 = 1, k2 = 0.5, B₀ = 0.2 µM; 5 s): A + B balance and the steady ratio.

**Hypothesis.** The ball's ~1.8% smooth-vs-staircase mismatch grows with surface-to-volume ratio and thin
features, so the uncorrected paths (native VCell; the co-sim with `none`) should degrade much
more on a dumbbell. `adapters+volumes` and the Netgen mesh should stay at the coupling-lag level.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `dumbbell` | `viva_pde_particle.composites.examples.bimolecular_hybrid` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_pde_particle.composites.examples.bimolecular_hybrid`** — `spec_viva_pde_particle_composites_examples_bimolecular_hybrid` (a plain, editable dict)


_composite spec file for `viva_pde_particle.composites.examples.bimolecular_hybrid` not found under `viva_pde_particle/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: dumbbell-three-ways ===
STUDY = 'dumbbell-three-ways'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**dumbbell-three-ways**


In [ ]:
# dumbbell-three-ways
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**dumbbell-3d**


In [ ]:
# dumbbell-3d
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**dumbbell-3d-movie**


In [ ]:
# dumbbell-3d-movie
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**dumbbell-3d-interactive**


In [ ]:
# dumbbell-3d-interactive
show_viz(_render_one('threejs:viz/dumbbell_3d_interactive.html', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| corrected-exchange-conserves | kind=derived_scalar field=b2f_vcell_corrected_exchange_balance | op range low -0.015 high 0.015 provenance {'kind': 'theory', 'note': 'First-order coupling error only.'} |
| corrected-near-membrane-unbiased | kind=derived_scalar field=b2f_vcell_corrected_band_bias | op range low -0.005 high 0.005 provenance {'kind': 'theory', 'note': 'E[B] is spatially uniform.'} |
| mesh-exchange-conserves | kind=derived_scalar field=b2f_mesh_exchange_balance | op range low -0.015 high 0.015 provenance {'kind': 'theory', 'note': 'The particle domain is the mesh (B2d).'} |
| native-exchange-conserves | kind=derived_scalar field=b2f_native_exchange_balance | op range low -0.015 high 0.015 provenance {'kind': 'theory', 'note': 'Kept so the native defect stays visible in the report card.'} |


## Study: Operator-splitting schemes for PDE/particle coupling: Jacobi (fvsolver), Gauss–Seidel, Strang (`splitting-schemes`)

**Question.** The embedded vcell-fvsolver hard-codes one lagged (Jacobi-type) splitting. With the
coupling expressed as process-bigraph processes, can other splittings be used, and how
much accuracy do they buy at a given coupling interval?

**Objective.** A5's A_p + B_f → C_f (k = 2/(µM·s), 20,000 A, 11×11×3 slab); Δt = 0.005;
τ = k·Δt with k ∈ {2, 4, 8, 16, 32}; 64 seeds per point.
Error: max over time of |mean survival − continuum|. Orders fitted for k ≥ 4.

**Hypothesis.** Jacobi and both Gauss–Seidel orderings are first order in the coupling interval τ.
Symmetric Strang splitting (half the PDE substeps, the particle step, the other half) is
second order in the mean dynamics and much more accurate at large τ.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `splitting-bimolecular` | `viva_pde_particle.composites.examples.bimolecular_hybrid` | 0 | k=2.0, dt=0.005 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_pde_particle.composites.examples.bimolecular_hybrid`** — `spec_viva_pde_particle_composites_examples_bimolecular_hybrid` (a plain, editable dict)


_composite spec file for `viva_pde_particle.composites.examples.bimolecular_hybrid` not found under `viva_pde_particle/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: splitting-schemes ===
STUDY = 'splitting-schemes'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**splitting-schemes**


In [ ]:
# splitting-schemes
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| jacobi-first-order | kind=derived_scalar field=b3_jacobi_order | op range low 0.7 high 1.5 provenance {'kind': 'theory', 'note': 'Explicit lagged splitting is first order.'} |
| gauss-seidel-first-order | kind=derived_scalar field=b3_gs_particles_first_order | op range low 0.7 high 1.5 provenance {'kind': 'theory', 'note': 'Sequential (Lie) splitting is first order.'} |
| strang-much-more-accurate | kind=derived_scalar field=b3_strang_over_jacobi_err_k32 | op < value 0.25 provenance {'kind': 'theory', 'note': 'Symmetric splitting cancels the leading splitting error.'} |


## Study: The hybrid model described in SBML Spatial, run on both solvers (`sbml-spatial-hybrid`)

**Question.** Can a PDE/particle hybrid be described in SBML Spatial, with each species' representation
(continuous or particle) as a VCell annotation, and run unchanged from that SBML on both
the native VCell solver and the co-simulation?

**Objective.** The A3 two-way exchange (11×11×3 slab, 10×10×1 µm; A_p → B at 1 s⁻¹, B → A_p at
0.5 s⁻¹·[B]; B₀ = 0.2 µM; dt = 0.01; 2 s; 8 seeds). `to_sbml` writes it to SBML Spatial
(A particle, B continuous); it is then run four ways:
- native from the original model (`to_biomodel` VCML) and native from the SBML (VCell's SBML
  import → libvcell → vcell-fvsolver);
- co-sim from the Python `HybridModel` and co-sim from `from_sbml`.

**Hypothesis.** The SBML carries everything the model needs (geometry, species, diffusion, initial
conditions, mass-action reactions, representations), so a run from the SBML reproduces the
run from the original model exactly, seed for seed, on each solver.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `two-way-exchange-from-sbml` | `viva_pde_particle.composites.examples.two_way_exchange` | 0 | k1=1.0, k2=0.5, b0=0.2, dt=0.01 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_pde_particle.composites.examples.two_way_exchange`** — `spec_viva_pde_particle_composites_examples_two_way_exchange` (a plain, editable dict)


_composite spec file for `viva_pde_particle.composites.examples.two_way_exchange` not found under `viva_pde_particle/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: sbml-spatial-hybrid ===
STUDY = 'sbml-spatial-hybrid'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**sbml-spatial-hybrid**


In [ ]:
# sbml-spatial-hybrid
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| model-round-trips-exactly | kind=derived_scalar field=b6_model_round_trip_exact | op == value 1 provenance {'kind': 'theory', 'note': "The SBML carries every model quantity; rate constants are recovered exactly from VCell's mass-action rate."} |
| cosim-from-sbml-is-identical | kind=derived_scalar field=b6_cosim_vs_cosim_sbml_identical | op == value 1 provenance {'kind': 'theory', 'note': 'Same HybridModel, same seeds.'} |
| native-from-sbml-is-identical | kind=derived_scalar field=b6_native_vs_native_sbml_identical | op == value 1 provenance {'kind': 'theory', 'note': "VCell's SBML import yields math equivalent to the original's (SBMLSpatialHybridTest), so the solver input is the same."} |
